# Data ingestion entry

1. Download
2. Validate
3. Save

In [1]:
# Imports

import pandas as pd
import numpy as np
import yfinance as yf
from datetime import date, timedelta
from vportfolio.params import DATA_START, UNIVERSE, BENCHMARK, RISK_FREE, RAW_FILE, MAX_DAILY_MOVE, LIVE_RAW
from vportfolio import data, pipelines

import sys
sys.path.append("..")
from tests.helpers import make_fake_raw, calm_fake_raw
import tempfile
from pathlib import Path

In [2]:
# Use yfinance libray to download them into a dataframe
    # threads set to false makes sure tickers are downloaded in sequence and not in parallel (less likely to be blocked by yahoo finance)
    # progress set to false hides the progress bar 
    # returns a dataframe with the prices of the listed tickers 
def data_download(end, start = DATA_START, universe = UNIVERSE, benchmark = BENCHMARK, rf = RISK_FREE):
    return yf.download(universe + benchmark + rf, start = start, end = end, auto_adjust = True, group_by = "column", threads = False, progress = False)

In [3]:
raw_data = data_download(date.today())

In [4]:
# last row should be Friday 13 November
raw_data.tail()

Price           Close                                                          \
Ticker            AGG        DBC        EEM         EFA        EMB       EMLC   
Date                                                                            
2026-11-09  96.580002  30.980000  61.029999  104.959999  92.540001  24.770000   
2026-11-10  95.879997  30.760000  60.570000  104.360001  91.980003  24.600000   
2026-11-11  95.839996  30.840000  62.040001  104.660004  92.070000  24.530001   
2026-11-12  95.220001  30.910000  61.139999  104.010002  91.339996  24.410000   
2026-11-13  95.089996  30.790001  60.919998  104.169998  91.129997  24.370001   

Price                                                     ...    Volume  \
Ticker            FXA         FXF        FXY         GLD  ...       LQD   
Date                                                      ...             
2026-11-09  68.599998  107.379997  58.040001  379.529999  ...  20997500   
2026-11-10  68.010002  106.620003  57.720001  369.359985  ...  28787100   
2026-11-11  67.809998  105.610001  57.290001  365.170013  ...   8811900   
2026-11-12  67.370003  105.349998  57.259998  354.890015  ...  21588500   
2026-11-13  67.070000  104.820000  57.380001  355.450012  ...  17958500   

Price                                                                     \
Ticker           QQQ       SHY       SPY        TIP       TLT        UUP   
Date                                                                       
2026-11-09  54905600   4789600  50281200  3955400.0  28945500  1984500.0   
2026-11-10  38845200   3831800  40698600  3432000.0  40461100  1077200.0   
2026-11-11  44871800   3778000  39151000  2802200.0  26568800  1167100.0   
2026-11-12  37219500  23964600  37460600  2386300.0  47053000   941900.0   
2026-11-13  41798500   3510500  38391200  2082500.0  31840900  1585300.0   

Price                                 
Ticker            VNQ       XLE ^IRX  
Date                                  
2026-11-09  5714800.0  40198400  0.0  
2026-11-10  2479900.0  33270500  0.0  
2026-11-11  3064800.0  24759500  0.0  
2026-11-12  4250400.0  26002000  0.0  
2026-11-13  2333900.0  18593200  0.0  

[5 rows x 115 columns]

In [5]:
# first date at or just after DATA_START = "2003-01-01"
raw_data.index[0]

Timestamp('2003-01-02 00:00:00')

In [6]:
# columns are same as DATA_KEYS = (["close", "open", "returns", "eligible", "rebal_dates", "cost_bp", "rf_daily"])
old = pd.read_parquet(RAW_FILE)
assert set(raw_data.columns) == set(old.columns)

In [7]:
# For the next function we need to establish a cap on the maximum daily move to filter data errors
processed = pipelines.data_pipeline(raw_data)
processed["returns"].abs().max().sort_values(ascending = False).head()
# a 30% treshold leaves enough space

Ticker
EEM    0.227699
XLE    0.201412
VNQ    0.195137
FXF    0.171697
EFA    0.158877
dtype: float64

In [8]:
# check if these deviations are real market days and not data errors
processed["returns"].abs().idxmax()[["EEM", "XLE", "VNQ", "FXF", "EFA"]]

Ticker
EEM   2008-10-13
XLE   2020-03-09
VNQ   2008-12-01
FXF   2015-01-15
EFA   2008-10-13
dtype: datetime64[s]

In [9]:
# Validate the downloaded data
    # will return a list of sentences, one per problem. Empty if all is fine
def validate_raw(raw, tickers = UNIVERSE + BENCHMARK, rf = RISK_FREE, start = DATA_START, max_move = MAX_DAILY_MOVE,
                 fields = ("Open", "High", "Low", "Close", "Volume")):
    # create a list of problems
    problems = []
    # Is every expected ticker there, for every expected field? If not, say which are missing
    expected_cols = pd.MultiIndex.from_product([fields, tickers + rf])
    missing_cols = expected_cols.difference(raw.columns)
    if not missing_cols.empty:
        problems.append(f"missing columns: {list(missing_cols)}")
        return problems  # the checks below would crash without the columns
    # Is the index in date order, with no date twice?
    if not raw.index.is_monotonic_increasing:
        problems.append("dates are not in order")
    if not raw.index.is_unique:
        problems.append("some dates appear twice")
    # Does the history start where it should (the first date is not much later than start)?
    first_date = raw.index[0]
    if first_date > pd.Timestamp(start) + pd.Timedelta(days=7):
        problems.append(f"history starts late: first date {first_date.date()}, expected {start}")
    # Is the last row complete: every universe and benchmark ticker has a close and an open?
    for field in ["Close", "Open"]:
        last_row = raw[field][tickers].iloc[-1]
        empty = last_row[last_row.isna()].index.tolist()
        if empty:
            problems.append(f"last row has no {field} for: {empty}")
    # Are all prices above zero?
    for field in ["Open", "High", "Low", "Close"]:
        bad = (raw[field][tickers] <= 0).any()
        bad_tickers = bad[bad].index.tolist()
        if bad_tickers:
            problems.append(f"{field} at or below zero for: {bad_tickers}")
    # Is the last day's move believable: no ticker moved more than some limit in one day?
    close = raw["Close"][tickers]
    last_move = (close.iloc[-1] / close.iloc[-2] - 1).abs()
    too_big = last_move[last_move > max_move].index.tolist()
    if too_big:
        problems.append(f"last day move above {max_move:.0%} for: {too_big}")
    return problems

In [10]:
# function trial
problems = validate_raw(raw_data)
assert problems == [], problems

In [11]:
make_fake_raw("2022-03-15").columns.get_level_values(0).unique()

Index(['Close', 'Open', 'High', 'Low', 'Volume'], dtype='str')

In [12]:
# function testing
def has(problems, text):
    return any(text in problem for problem in problems)

raw = calm_fake_raw()
start = "2019-01-01" # the fake history starts in 2019, not at DATA_START
last = raw.index[-1]
# 1. always true: clean data gives no problems
assert validate_raw(raw, start = start) == []
# 2. known answer: break one thing at a time, that problem is reported
no_ticker = raw.drop(columns = [("Close", "EEM")])
assert has(validate_raw(no_ticker, start = start), "missing columns")
twice = pd.concat([raw, raw.iloc[[-1]]])
assert has(validate_raw(twice, start = start), "appear twice")
reversed_dates = raw.iloc[::-1]
assert has(validate_raw(reversed_dates, start = start), "not in order")
late = validate_raw(raw, start = "2010-01-01")
assert len(late) == 1 and has(late, "history starts late")
no_close = raw.copy()
no_close.loc[last, ("Close", "TLT")] = np.nan
problems = validate_raw(no_close, start = start)
assert len(problems) == 1 and has(problems, "last row has no Close")
zero_price = raw.copy()
zero_price.loc[raw.index[5], ("Low", "XLE")] = 0
problems = validate_raw(zero_price, start = start)
assert len(problems) == 1 and has(problems, "Low at or below zero")
jump = raw.copy()
jump.loc[last, ("Close", "UUP")] *= 3
problems = validate_raw(jump, start = start)
assert len(problems) == 1 and has(problems, "last day move")
# 3. inputs matter: a 10% move passes a loose limit and fails a tight one
move = raw.copy()
move.loc[last, ("Close", "EEM")] = raw["Close"]["EEM"].iloc[-2] * 1.10
assert validate_raw(move, start = start, max_move = 0.30) == []
assert has(validate_raw(move, start = start, max_move = 0.05), "last day move")
# 3. inputs matter: the rate ticker may be empty on the last row (bond holiday)
no_rate = raw.copy()
for field in ["Open", "High", "Low", "Close", "Volume"]:
    no_rate.loc[last, (field, "^IRX")] = np.nan
assert validate_raw(no_rate, start = start) == []

In [13]:
# Save the validated raw_data into a parquet file
def save_data(raw_data, run_date, folder = LIVE_RAW):
    folder.mkdir(parents = True, exist_ok = True)
    path = folder / f"yahoo_prices_{run_date}.parquet"
    if not path.exists():
        raw_data.to_parquet(path)
    return path

In [14]:
# function trial
path = save_data(raw_data, date.today())
path

PosixPath('/home/antonio/code/pollypineapple1/gcp/ML_virtual_portfolio/data/raw/live/yahoo_prices_2026-11-16.parquet')

In [15]:
# function testing
tmp_path = Path(tempfile.mkdtemp())

raw = make_fake_raw("2022-03-15")
path = save_data(raw, "2026-12-01", tmp_path)
# 1. always true: the file exists, in the folder we passed
assert path.exists() and path.parent == tmp_path
# 2. known answer: the name carries the run date, and reading it back gives the same table
assert path.name == "yahoo_prices_2026-12-01.parquet"
pd.testing.assert_frame_equal(pd.read_parquet(path), raw, check_freq = False)
# 3. inputs matter: another date makes another file
other_path = save_data(raw, "2026-12-02", tmp_path)
assert other_path != path and len(list(tmp_path.iterdir())) == 2
# 3. inputs matter: saving a different table on the same date keeps the first file
save_data(raw * 2, "2026-12-01", tmp_path)
pd.testing.assert_frame_equal(pd.read_parquet(path), raw, check_freq = False)

Now we chain the steps into a pipeline